In [1]:
!pip install -q -U transformers datasets accelerate peft sentence-transformers wandb 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, gc, re, random, warnings
import numpy as np, pandas as pd, torch

In [ ]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    print("W&B logged in via secret")
except Exception as e:
    print("Secret not found, will prompt interactively:", e)

In [ ]:

warnings.filterwarnings("ignore")

class CFG:
    # --- data ---
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"  # change if your dataset folder name differs
    OPTS     = ["A", "B", "C", "D", "E"]                   # the five answer choices
    SEED     = 42                                          # fixes randomness -> reproducible

    # --- score-driver model (DeBERTa + LoRA, Milestone 4) ---
    MODEL_NAME = "microsoft/deberta-v3-base"
    MAX_LEN    = 256      # max tokens per (question + one option) pair
    EPOCHS     = 3        # passes over the training data
    LR         = 2e-4     # LoRA uses a higher learning rate than full fine-tuning
    TRAIN_BS   = 4        # training batch size
    EVAL_BS    = 8        # evaluation batch size

    # --- pretrained zero-shot model (Milestone 2) ---
    ST_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"

    # --- Weights & Biases ---
    WANDB_PROJECT = "23f2002523-t22026"   # <-- put YOUR W&B project name here

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(CFG.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("torch:", torch.__version__)

In [ ]:
train = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
samp  = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train shape:", train.shape)   # (rows, columns)
print("test shape :", test.shape)
print("columns    :", list(train.columns))

# how often each letter is the correct answer (checks for imbalance)
print("\nAnswer counts:")
print(train["answer"].value_counts().sort_index())

# check for any missing/empty cells
print("\nMissing values -> train:", train.isnull().sum().sum(),
      "| test:", test.isnull().sum().sum())

# peek at one full example
print("\n--- Example row 0 ---")
print("PROMPT:", train.loc[0, "prompt"][:200])
for o in CFG.OPTS:
    print(f"{o}:", str(train.loc[0, o])[:120])
print("ANSWER:", train.loc[0, "answer"])

train.head(2)

In [ ]:
from sklearn.metrics import f1_score

def map_at_3(true_letters, pred_lists):
    """
    true_letters: list of correct letters, e.g. ["B", "A", ...]
    pred_lists:   list of ranked top-3 guesses, e.g. [["B","C","A"], ...]
    Returns the mean MAP@3 score.
    """
    total = 0.0
    for gt, preds in zip(true_letters, pred_lists):
        for rank, p in enumerate(preds[:3]):     # rank = 0,1,2
            if p == gt:
                total += 1.0 / (rank + 1)         # 1, 1/2, or 1/3
                break                             # stop at first match
    return total / len(true_letters)

def probs_to_top3(probs):
    """probs: array (N,5) of scores per option -> list of ranked top-3 letter lists."""
    order = np.argsort(-probs, axis=1)[:, :3]     # indices of 3 highest scores
    return [[CFG.OPTS[j] for j in row] for row in order]

def evaluate(true_letters, probs):
    """Returns map@3, accuracy, and macro-F1 in one dict (used for W&B comparison)."""
    top3   = probs_to_top3(probs)
    y_true = np.array([CFG.OPTS.index(a) for a in true_letters])
    y_pred = probs.argmax(1)
    return {
        "map@3":    round(map_at_3(true_letters, top3), 4),
        "accuracy": round(float((y_pred == y_true).mean()), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, average="macro")), 4),
    }

# quick self-test so we trust the function
test_true = ["A", "B"]
test_pred = [["A", "X", "Y"],   # correct in 1st place -> 1.0
             ["X", "B", "Y"]]   # correct in 2nd place -> 0.5
print("Self-test MAP@3 (should be 0.75):", map_at_3(test_true, test_pred))

In [ ]:
# templated wrappers to strip (case-insensitive)
PREFIX = re.compile(r"^\s*(pick the best possible answer|choose the correct.*?|"
                    r"select the (?:correct|best).*?|identify the.*?)\s*:\s*", re.I)
SUFFIX = re.compile(r"\s*(among the listed options|from the following choices|"
                    r"from the options|carefully)\.?\s*$", re.I)

def clean_text(s):
    s = str(s).strip()          # force string, trim spaces
    s = PREFIX.sub("", s)       # drop leading template
    s = SUFFIX.sub("", s)       # drop trailing template
    return re.sub(r"\s+", " ", s).strip()   # collapse extra spaces

# apply to both train and test; clean the prompt AND the options
for df in (train, test):
    df["prompt_clean"] = df["prompt"].map(clean_text)
    for o in CFG.OPTS:
        df[o] = df[o].map(clean_text)

# before / after check
print("BEFORE:", repr(train.loc[0, "prompt"]))
print("AFTER :", repr(train.loc[0, "prompt_clean"]))

In [ ]:
from collections import Counter

def simple_tokenize(text):
    """Lowercase and split into word/number tokens."""
    return re.findall(r"[a-z0-9]+", str(text).lower())

# 1) count all words in the training prompts + options
counter = Counter()
for col in ["prompt_clean"] + CFG.OPTS:
    for text in train[col]:
        counter.update(simple_tokenize(text))

# 2) build vocab: reserve 0 for padding, 1 for unknown words
vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in counter.most_common(20000):   # keep 20k most frequent words
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

# 3) function to turn a sentence into a fixed-length list of IDs
SCRATCH_MAXLEN = 80   # each text becomes exactly 80 tokens

def encode(text, maxlen=SCRATCH_MAXLEN):
    ids = [vocab.get(w, 1) for w in simple_tokenize(text)]  # 1 = <unk> for unseen words
    ids = ids[:maxlen]                                      # cut if too long
    ids = ids + [0] * (maxlen - len(ids))                   # pad with 0 if too short
    return ids

# quick check
print("Example encoding of 'quantum physics is hard':")
print(encode("quantum physics is hard")[:15], "...")

In [ ]:
import torch.nn as nn

class ScratchBiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        # 1) Embedding: turns each word ID into a learnable 128-number vector
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # 2) BiGRU: reads the sequence forwards AND backwards to capture context
        self.gru = nn.GRU(embed_dim, embed_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)   # regularization: prevents overfitting
        # 3) Scorer: takes [prompt_vec, option_vec] and outputs one number (the score)
        self.scorer = nn.Sequential(
            nn.Linear(embed_dim * 4, embed_dim),  # *4 because BiGRU doubles, and we concat 2 texts
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 1),
        )

    def encode(self, x):
        """Turn a batch of token-ID sequences into summary vectors (mean-pooled BiGRU)."""
        mask = (x != 0).float().unsqueeze(-1)      # ignore padding positions
        emb = self.embedding(x)
        out, _ = self.gru(emb)                     # (batch, seq_len, 2*embed_dim)
        summed = (out * mask).sum(1)               # sum only real tokens
        return summed / mask.sum(1).clamp(min=1)   # average -> one vector per text

    def forward(self, prompt_ids, option_ids):
        # prompt_ids: (B, L)   option_ids: (B, 5, L)
        B, C, L = option_ids.shape
        p_vec = self.encode(prompt_ids)                          # (B, 2*embed)
        o_vec = self.encode(option_ids.reshape(B*C, L)).reshape(B, C, -1)  # (B,5,2*embed)
        p_rep = p_vec.unsqueeze(1).expand(-1, C, -1)             # repeat prompt for each option
        combined = torch.cat([p_rep, o_vec], dim=-1)            # (B,5,4*embed)
        scores = self.scorer(self.dropout(combined)).squeeze(-1)  # (B,5)
        return scores

# build it and count parameters
scratch_model = ScratchBiGRU(len(vocab)).to(DEVICE)
n_params = sum(p.numel() for p in scratch_model.parameters())
print("Model built. Trainable parameters:", f"{n_params:,}")

In [ ]:
from sklearn.model_selection import train_test_split

# 1) 85/15 split, stratified so answer letters stay balanced in both parts
train_df, val_df = train_test_split(
    train, test_size=0.15, random_state=CFG.SEED, stratify=train["answer"]
)
print("Train rows:", len(train_df), "| Val rows:", len(val_df))

# 2) helper: turn a dataframe into (prompt_tensor, options_tensor, label_tensor)
def make_tensors(df):
    P = torch.tensor([encode(t) for t in df["prompt_clean"]], dtype=torch.long)
    O = torch.stack([torch.tensor([encode(t) for t in df[o]], dtype=torch.long)
                     for o in CFG.OPTS], dim=1)          # (N, 5, L)
    y = torch.tensor([CFG.OPTS.index(a) for a in df["answer"]], dtype=torch.long)
    return P, O, y

P_tr, O_tr, y_tr = make_tensors(train_df)
P_va, O_va, y_va = make_tensors(val_df)
P_tr, O_tr, y_tr = P_tr.to(DEVICE), O_tr.to(DEVICE), y_tr.to(DEVICE)
P_va, O_va, y_va = P_va.to(DEVICE), O_va.to(DEVICE), y_va.to(DEVICE)

# 3) train
import wandb
run = wandb.init(project=CFG.WANDB_PROJECT, name="model1_scratch_bigru", reinit=True)

optimizer = torch.optim.Adam(scratch_model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.CrossEntropyLoss()
EPOCHS_SCRATCH = 12
BATCH = 64

for epoch in range(EPOCHS_SCRATCH):
    scratch_model.train()
    perm = torch.randperm(len(P_tr), device=DEVICE)   # shuffle each epoch
    for i in range(0, len(P_tr), BATCH):
        idx = perm[i:i+BATCH]
        scores = scratch_model(P_tr[idx], O_tr[idx])
        loss = loss_fn(scores, y_tr[idx])
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    # evaluate on validation each epoch
    scratch_model.eval()
    with torch.no_grad():
        val_scores = scratch_model(P_va, O_va)
        val_probs = torch.softmax(val_scores, dim=1).cpu().numpy()
    m = evaluate(val_df["answer"].tolist(), val_probs)
    print(f"epoch {epoch+1:2d} | train_loss {loss.item():.3f} | val {m}")
    wandb.log({"epoch": epoch+1, "train_loss": loss.item(), **{f"val_{k}": v for k, v in m.items()}})

# save this model's validation predictions for the ensemble later
scratch_val_probs = val_probs
scratch_metrics = m
wandb.finish()
print("\nFinal from-scratch validation:", scratch_metrics)

In [ ]:
# 1) encode the test data (no labels here — Kaggle has them hidden)
P_te = torch.tensor([encode(t) for t in test["prompt_clean"]], dtype=torch.long).to(DEVICE)
O_te = torch.stack([torch.tensor([encode(t) for t in test[o]], dtype=torch.long)
                    for o in CFG.OPTS], dim=1).to(DEVICE)

# 2) predict with the trained from-scratch model
scratch_model.eval()
with torch.no_grad():
    test_scores = scratch_model(P_te, O_te)
    test_probs = torch.softmax(test_scores, dim=1).cpu().numpy()

# 3) convert to ranked top-3 letters and build the submission file
top3 = probs_to_top3(test_probs)
submission = pd.DataFrame({
    "ID": test["id"].values,
    "Prediction": [" ".join(t) for t in top3],
})

# 4) safety checks: exact format match with sample_submission
assert list(submission.columns) == list(samp.columns), "column names don't match!"
assert len(submission) == len(test), "row count doesn't match!"
assert submission["Prediction"].str.split().map(len).eq(3).all(), "each row needs 3 letters!"

submission.to_csv("submission.csv", index=False)
print("submission.csv written:", submission.shape)
submission.head()

In [ ]:
class PromptBlindModel(nn.Module):
    """Scores each option using ONLY the option's own text. No prompt input at all."""
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, embed_dim, batch_first=True, bidirectional=True)
        self.scorer = nn.Sequential(nn.Linear(embed_dim*2, embed_dim), nn.ReLU(),
                                    nn.Linear(embed_dim, 1))
    def encode(self, x):
        mask = (x != 0).float().unsqueeze(-1)
        out, _ = self.gru(self.embedding(x))
        return (out * mask).sum(1) / mask.sum(1).clamp(min=1)
    def forward(self, option_ids):               # note: NO prompt argument!
        B, C, L = option_ids.shape
        o_vec = self.encode(option_ids.reshape(B*C, L)).reshape(B, C, -1)
        return self.scorer(o_vec).squeeze(-1)     # (B, 5)

# train it on the SAME 85/15 split (options only — prompts never enter the model)
blind_model = PromptBlindModel(len(vocab)).to(DEVICE)
optimizer = torch.optim.Adam(blind_model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(10):
    blind_model.train()
    perm = torch.randperm(len(O_tr), device=DEVICE)
    for i in range(0, len(O_tr), 64):
        idx = perm[i:i+64]
        loss = loss_fn(blind_model(O_tr[idx]), y_tr[idx])
        optimizer.zero_grad(); loss.backward(); optimizer.step()

blind_model.eval()
with torch.no_grad():
    blind_probs = torch.softmax(blind_model(O_va), dim=1).cpu().numpy()
blind_metrics = evaluate(val_df["answer"].tolist(), blind_probs)
print("PROMPT-BLIND model (never sees the question) ->", blind_metrics)

In [ ]:
print("="*62)
print("ARTIFACT EXPERIMENT — SUMMARY OF EVIDENCE")
print("="*62)
print(f"""
Hypothesis: correct answers carry a stylistic 'fingerprint'
that identifies them WITHOUT reading the question.

Evidence 1 — Prompt-blind model:
  A model given ONLY the 5 option texts (question completely
  hidden) reaches:
      MAP@3    = {blind_metrics['map@3']}
      accuracy = {blind_metrics['accuracy']}
  Random guessing would give accuracy ~0.20, MAP@3 ~0.37.
  ==> The answer is identifiable from option text alone.

Evidence 2 — Leaderboard transfer test:
  A diagnostic Kaggle submission of the from-scratch model
  scored 0.75311 on the public leaderboard.
  ==> The fingerprint also exists in the hidden test set
      (train and test were generated by the same process).

Conclusion:
  The dataset contains an annotation artifact (a known ML
  phenomenon, cf. hypothesis-only artifacts in SNLI).
  Therefore validation scores can overstate true reasoning
  ability. My final solution combines this signal with a
  fine-tuned DeBERTa that reasons over (question + option)
  pairs, rather than relying on the artifact alone.
""")

In [ ]:
from sentence_transformers import SentenceTransformer

# load the pretrained model (downloads ~90MB first time)
st_model = SentenceTransformer(CFG.ST_MODEL, device=DEVICE)

def minilm_probs(df):
    """Embed prompt + all 5 options, rank options by cosine similarity to the prompt."""
    p_emb = st_model.encode(df["prompt_clean"].tolist(), normalize_embeddings=True,
                            convert_to_numpy=True, batch_size=64, show_progress_bar=False)
    sims = np.zeros((len(df), 5), dtype=np.float32)
    for i, o in enumerate(CFG.OPTS):
        o_emb = st_model.encode(df[o].tolist(), normalize_embeddings=True,
                                convert_to_numpy=True, batch_size=64, show_progress_bar=False)
        sims[:, i] = (p_emb * o_emb).sum(axis=1)      # cosine sim (vectors are normalized)
    # softmax with temperature -> probability-like scores for fair comparison/ensembling
    e = np.exp((sims - sims.max(axis=1, keepdims=True)) / 0.1)
    return e / e.sum(axis=1, keepdims=True)

# evaluate on the SAME validation split as Model 1 (fair comparison)
minilm_val_probs = minilm_probs(val_df)
minilm_metrics = evaluate(val_df["answer"].tolist(), minilm_val_probs)
print("Model 2 (pretrained MiniLM, zero-shot) ->", minilm_metrics)

# log to W&B as its own run
run = wandb.init(project=CFG.WANDB_PROJECT, name="model2_pretrained_minilm", reinit=True)
wandb.log(minilm_metrics)
wandb.finish()

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

# DeBERTa ka apna tokenizer (hamare from-scratch wale se bahut zyada powerful -
# subword tokenization, 128k vocab, unknown words ki problem nahi)
deberta_tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)

def preprocess(example):
    """Ek question -> 5 (question, option) pairs, tokenized."""
    first  = [example["prompt_clean"]] * 5            # question, 5 baar repeat
    second = [example[o] for o in CFG.OPTS]           # paanchon options
    tok = deberta_tokenizer(first, second,            # pair-tokenization: [CLS] q [SEP] opt [SEP]
                            truncation=True, max_length=CFG.MAX_LEN)
    out = {k: v for k, v in tok.items()}              # input_ids, attention_mask (each = 5 lists)
    if "label" in example:
        out["label"] = example["label"]
    return out

# label column: answer letter -> number (A=0, B=1, ...)
train_df = train_df.copy(); val_df = val_df.copy()
train_df["label"] = train_df["answer"].map({l: i for i, l in enumerate(CFG.OPTS)})
val_df["label"]   = val_df["answer"].map({l: i for i, l in enumerate(CFG.OPTS)})

# pandas -> HuggingFace Dataset -> tokenize sab rows
cols = ["prompt_clean"] + CFG.OPTS + ["label"]
train_ds = Dataset.from_pandas(train_df[cols], preserve_index=False).map(preprocess, remove_columns=cols)
val_ds   = Dataset.from_pandas(val_df[cols],   preserve_index=False).map(preprocess, remove_columns=cols)

print("train_ds:", train_ds)
print("\nEk example ki shape check:")
ex = train_ds[0]
print("input_ids: 5 sequences?", len(ex["input_ids"]) == 5)
print("pehli sequence ke first 12 tokens:", ex["input_ids"][0][:12])
print("label:", ex["label"])

In [ ]:
import os, wandb
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
if wandb.run is not None:
    wandb.finish()

# Trainer ko W&B se poori tarah alag karo
trainer.args.report_to = []
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if "Wandb" not in type(cb).__name__
]

# ---- 1) agreement check (in-memory, no logging) ----
deberta_preds = deberta_val_probs.argmax(1) if 'deberta_val_probs' in dir() else None
if deberta_preds is None:
    # agar val probs nahi bane the, abhi banao
    val_logits = trainer.predict(val_ds).predictions
    deberta_val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
    deberta_preds = deberta_val_probs.argmax(1)
    print("Val metrics:", evaluate(val_df["answer"].tolist(), deberta_val_probs))

blind_preds = blind_probs.argmax(1)
print(f"DeBERTa-vs-Blind agreement: {(deberta_preds == blind_preds).mean():.1%}")

# ---- 2) test predictions ----
test_cols = ["prompt_clean"] + CFG.OPTS
test_ds = Dataset.from_pandas(test[test_cols], preserve_index=False).map(
    preprocess, remove_columns=test_cols)
test_logits = trainer.predict(test_ds).predictions
deberta_test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()

# ---- 3) submission ----
top3 = probs_to_top3(deberta_test_probs)
submission = pd.DataFrame({"ID": test["id"].values,
                           "Prediction": [" ".join(t) for t in top3]})
submission.to_csv("submission.csv", index=False)
print("submission.csv written:", submission.shape)
submission.head()